# CSE440 Lab Project: Multi-Class Text Classification
## A Comparison of Word Representations and ML/NN Models

**Objective:** To classify Question-Answer text into categories using various Machine Learning and Deep Learning models, comparing TF-IDF and Skip-gram word representations.

**Dataset:** Assigned Project Dataset (train.csv, test.csv)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Embedding, SimpleRNN, GRU, LSTM, Bidirectional, GlobalAveragePooling1D, Input
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from gensim.models import Word2Vec

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


## 1. Load Data
Loading the training and testing datasets.


In [ ]:
# Load datasets
try:
    df_train = pd.read_csv('train.csv')
    df_test = pd.read_csv('test.csv')
    print("Datasets loaded successfully.")
    print(f"Train shape: {df_train.shape}")
    print(f"Test shape: {df_test.shape}")
except Exception as e:
    print(f"Error loading data: {e}")

# Display first few rows
df_train.head()


## 2. Exploratory Data Analysis (EDA)
Analyzing class distribution and text characteristics.


In [ ]:
# Check for null values
print(df_train.isnull().sum())
df_train.dropna(inplace=True)
df_test.dropna(inplace=True)

# Class Distribution
plt.figure(figsize=(12, 6))
sns.countplot(y=df_train['Class'], order=df_train['Class'].value_counts().index)
plt.title('Class Distribution in Training Set')
plt.xlabel('Count')
plt.ylabel('Class')
plt.show()


## 3. Data Preprocessing
We apply the following preprocessing steps:
1. **HTML Removal**: To clean the raw web-scraped text.
2. **Regex Cleaning**: Removing non-alphabetic characters.
3. **Lowercasing**: For normalization.
4. **Stopword Removal**: To remove non-informative words.
5. **Lemmatization**: To reduce words to their base form.


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', str(text))
    # Remove non-alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Lowercase
    text = text.lower()
    # Tokenize
    tokens = text.split()
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)

# Apply cleaning
print("Preprocessing training data...")
df_train['clean_text'] = df_train['QA Text'].apply(clean_text)
print("Preprocessing testing data...")
df_test['clean_text'] = df_test['QA Text'].apply(clean_text)

df_train.head()


### Label Encoding
Converting string labels to integers.


In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(df_train['Class'])
y_test = le.transform(df_test['Class'])

num_classes = len(le.classes_)
print(f"Classes: {le.classes_}")
print(f"Number of classes: {num_classes}")

# One-hot encoding for NN models
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)


## 4. Experiment Set 1: TF-IDF Representations
Using TF-IDF with Random Forest (ML) and a Deep Neural Network (DNN).


In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(df_train['clean_text']).toarray()
X_test_tfidf = tfidf.transform(df_test['clean_text']).toarray()

print(f"TF-IDF Matrix Shape: {X_train_tfidf.shape}")


### Model 1: Random Forest (ML Baseline)


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_tfidf, y_train)
y_pred_rf = rf_model.predict(X_test_tfidf)

print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))


### Model 2: Deep Neural Network (DNN) with TF-IDF


In [ ]:
def build_dnn_tfidf(input_dim, num_classes):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

dnn_tfidf = build_dnn_tfidf(X_train_tfidf.shape[1], num_classes)
history_dnn_tfidf = dnn_tfidf.fit(X_train_tfidf, y_train_cat, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Evaluation
loss, acc = dnn_tfidf.evaluate(X_test_tfidf, y_test_cat)
print(f"DNN (TF-IDF) Accuracy: {acc}")


## 5. Experiment Set 2: Skip-gram (Word2Vec) Representations
Training a Word2Vec model using the Skip-gram approach (`sg=1`) and using it for all Neural Network models.


In [ ]:
# Tokenize for Word2Vec
X_train_tokens = [text.split() for text in df_train['clean_text']]

# Train Word2Vec Skip-gram
w2v_size = 100
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=w2v_size, window=5, min_count=1, workers=4, sg=1)
print("Word2Vec model trained.")

# Create Embedding Matrix
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_train['clean_text'])
vocab_size = len(tokenizer.word_index) + 1

embedding_matrix = np.zeros((vocab_size, w2v_size))
for word, i in tokenizer.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]

print(f"Embedding Matrix Shape: {embedding_matrix.shape}")


In [ ]:
# Prepare Sequences
max_len = 100  # Define max sequence length
X_train_seq = tokenizer.texts_to_sequences(df_train['clean_text'])
X_test_seq = tokenizer.texts_to_sequences(df_test['clean_text'])

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)


### Training Helper Function


In [ ]:
def train_evaluate_model(model, name):
    print(f"\nTRAINING {name}...")
    history = model.fit(X_train_pad, y_train_cat, epochs=10, batch_size=64, validation_split=0.2, verbose=1)
    
    print(f"\nEVALUATING {name}...")
    loss, acc = model.evaluate(X_test_pad, y_test_cat, verbose=0)
    y_pred_probs = model.predict(X_test_pad)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    print(f"{name} Test Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    # Plot History
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.title(f'{name} Accuracy')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f'{name} Loss')
    plt.legend()
    plt.show()
    
    return acc


### Model 3: DNN with Skip-gram (Averaged Embeddings)
Since standard DNNs take fixed-size input, we use Global Average Pooling on the embeddings.


In [ ]:
def build_dnn_w2v():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        GlobalAveragePooling1D(),
        Dense(256, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_dnn_w2v = train_evaluate_model(build_dnn_w2v(), "DNN (Skip-gram)")


### Model 4: SimpleRNN


In [ ]:
def build_rnn():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        SimpleRNN(128),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_rnn = train_evaluate_model(build_rnn(), "SimpleRNN")


### Model 5: GRU


In [ ]:
def build_gru():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        GRU(128),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_gru = train_evaluate_model(build_gru(), "GRU")


### Model 6: LSTM


In [ ]:
def build_lstm():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        LSTM(128),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_lstm = train_evaluate_model(build_lstm(), "LSTM")


### Model 7: Bidirectional SimpleRNN


In [ ]:
def build_bi_rnn():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        Bidirectional(SimpleRNN(128)),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_bi_rnn = train_evaluate_model(build_bi_rnn(), "Bidirectional RNN")


### Model 8: Bidirectional GRU


In [ ]:
def build_bi_gru():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        Bidirectional(GRU(128)),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_bi_gru = train_evaluate_model(build_bi_gru(), "Bidirectional GRU")


### Model 9: Bidirectional LSTM


In [ ]:
def build_bi_lstm():
    model = Sequential([
        Embedding(vocab_size, w2v_size, weights=[embedding_matrix], input_length=max_len, trainable=False),
        Bidirectional(LSTM(128)),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

acc_bi_lstm = train_evaluate_model(build_bi_lstm(), "Bidirectional LSTM")


## 6. Conclusion & Comparison
Summary of all model performances.


In [ ]:
results = {
    'Model': ['Random Forest (TF-IDF)', 'DNN (TF-IDF)', 'DNN (Skip-gram)', 'SimpleRNN', 'GRU', 'LSTM', 'Bi-RNN', 'Bi-GRU', 'Bi-LSTM'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_rf),
        dnn_tfidf.evaluate(X_test_tfidf, y_test_cat, verbose=0)[1],
        acc_dnn_w2v,
        acc_rnn,
        acc_gru,
        acc_lstm,
        acc_bi_rnn,
        acc_bi_gru,
        acc_bi_lstm
    ]
}

results_df = pd.DataFrame(results)
results_df.sort_values(by='Accuracy', ascending=False, inplace=True)
print(results_df)

plt.figure(figsize=(10, 6))
sns.barplot(x='Accuracy', y='Model', data=results_df, palette='viridis')
plt.title('Model Comparison')
plt.xlim(0, 1)
plt.show()
